# IMEN266 · Ch.7 · Class Problems 9 (part 1) — Renewal processes: counting, renewal function, limit theorems, Wald

**How this notebook works** — every problem follows the same 5 steps:

> **Problem → Model → Analytic → Simulate & Visualize → Experiment**

The analytic answer and the simulation must agree. *That agreement — not the
instructor, not an AI — is your ground truth.*

**In-class version only:** cells marked `# TODO` are for you to complete during
class. A `check(...)` cell right after tells you immediately whether your answer
matches the reference value.

▶ Open in Colab: `https://colab.research.google.com/github/youngmko/imen266-2026/blob/main/ch07/notebooks/CP9_renewal_limits.ipynb`

**The facts of this notebook** *(slides pp. 1–29/54; Companion Proofs 1–4)*

- A **renewal process** counts events whose gaps $X_1, X_2, \dots$ are iid with
  distribution $F$ (mean $\mu$): $S_n = X_1+\dots+X_n$, $N(t)=\max\{n: S_n\le t\}$,
  and $N(t)\ge n \Leftrightarrow S_n\le t$, so $P(N(t)=n)=F_n(t)-F_{n+1}(t)$.
- The **renewal function** $m(t)=E[N(t)]=\sum_n F_n(t)$ solves the **renewal
  equation** $m(t)=F(t)+\int_0^t m(t-x)\,dF(x)$ (uniform$(0,1)$: $m(t)=e^t-1$ on $[0,1]$).
- **Limit theorems:** $N(t)/t\to 1/\mu$ a.s. and $m(t)/t\to 1/\mu$ — only the mean
  of $F$ sets the rate; the shape sets how fast the rate settles.
- **Wald's equation:** $E[\sum_{n=1}^N X_n]=E[N]E[X]$ when $N$ is a stopping time
  with $E[N]<\infty$; $N(t)+1$ is a stopping time, $N(t)$ is not.

In code (course package `imen266.renewal`): `simulate_renewal(sampler, T, rng)`,
`count(epochs, t)`, `renewal_function(dist, t)` (discretised renewal equation),
`renewal_function_mc`, closed forms `m_uniform01`, `m_hyperexp`, `m_erlang2`,
`samplers_same_mean(mean)` (six gap distributions with the same mean), and
`rate_convergence_plot`. A *sampler* is any `f(rng, size)` returning iid gaps.

In [ ]:
# --- IMEN266 setup (Ch.7): course package + helpers ---------------------------
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

try:                                   # local clone / PLMS zip: package is on the path
    from imen266.renewal import simulate_renewal
except ImportError:                    # Colab: fetch the course package from GitHub (~5 s)
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/youngmko/imen266-2026.git"], check=True)
    from imen266.renewal import simulate_renewal
from imen266.renewal import (hyperexp, sampler_from, samplers_same_mean, count,
                             renewal_function, renewal_function_series, renewal_function_mc,
                             m_uniform01, m_hyperexp, m_erlang2, rate_convergence_plot,
                             renewal_reward, age_residual, age_residual_samples,
                             equilibrium_cdf, length_biased_pdf, inspection_paradox_demo,
                             alternating_renewal, semi_markov)
from imen266.ctmc import poisson_process, CTMC, machine_repair
from imen266.check import check        # [TODO]/[OK]/[X] self-check (unfinished TODOs never crash)

rng = np.random.default_rng(2026)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True,
                     "grid.alpha": .3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11,
                     "legend.frameon": False})

def plot_counting(epochs, T, ax=None, **kw):
    """Step plot of N(t) on [0, T] from the renewal epochs S_1 < S_2 < ... ."""
    ax = ax or plt.gca()
    S = np.asarray(epochs); S = S[S <= T]
    ax.step(np.concatenate([[0.0], S, [T]]), np.concatenate([[0], np.arange(1, len(S) + 1), [len(S)]]), where="post", **kw)
    ax.set_xlabel("t"); ax.set_ylabel("N(t)")
    return ax

def plot_sawtooth(epochs, T, ax=None):
    """Age A(t) (rising) and remaining time Y(t) (falling) on [0, T]; epochs must extend beyond T."""
    ax = ax or plt.gca()
    ts = np.linspace(0, T, 2000)
    A, Y, _ = age_residual(epochs, ts)
    ax.plot(ts, A, lw=1.2, label="age A(t)"); ax.plot(ts, Y, lw=1.2, label="remaining Y(t)")
    ax.set_xlabel("t"); ax.legend(fontsize=8)
    return ax

try:
    from ipywidgets import interact, FloatSlider, IntSlider, FloatLogSlider
    HAS_W = True
except Exception:
    HAS_W = False

def show(fn, **sliders):
    """interact() if widgets are available; otherwise draw once with defaults."""
    if HAS_W:
        interact(fn, **sliders)
    else:
        fn(**{k: getattr(v, "value", v) for k, v in sliders.items()})

print("Setup OK.  imen266.renewal imported.  Widgets available:", HAS_W)

---
## Warm-up · Same mean gap, different regularity *(slides pp. 1–4/54)*

Six renewal processes whose gaps all have mean 1: deterministic, uniform$(0,2)$,
Erlang(4), exponential (the Poisson process), hyperexponential and lognormal.
Watch the counting paths $N(t)$ on a short horizon, then the rates $N(T)/T$ on a
long one. The Poisson process is the only one that "restarts" at every time;
the others restart only at their event times (slide p. 4/54).

In [ ]:
T_show, T_long = 30, 20_000
samplers = samplers_same_mean(1.0)                       # name -> sampler f(rng, size), all with mean 1

fig, ax = plt.subplots(figsize=(9, 4))
for name, f in samplers.items():
    S = simulate_renewal(f, T_show, rng)
    plot_counting(S, T_show, ax, lw=1.2, label=f"{name} (N(30) = {len(S)})")
ax.plot([0, T_show], [0, T_show], "k--", lw=1, label="t / mu")
ax.legend(fontsize=8); ax.set_title("Same mean gap: the regular processes hug t/mu, the bursty ones wander"); plt.show()

print(f"long run, T = {T_long}:  N(T)/T   (every process: 1/mu = 1)")
for name, f in samplers.items():
    S = simulate_renewal(f, T_long, rng)
    gaps = np.diff(np.concatenate([[0.0], S]))
    print(f"  {name:>18}: N(T)/T = {len(S)/T_long:.4f}   gap cv = {gaps.std()/gaps.mean():.2f}")

**Think about it** — (i) Which of the six paths could be observed from *any*
starting time without changing its law, and why? (ii) At $t=30$ the deterministic
path has exactly $N=30$; the others deviate by a few units. The deviation grows
like $\sigma\sqrt{t}/\mu^{3/2}$ (a central limit theorem for $N(t)$) — read off the
gap cv's printed above and predict which process is the noisiest at $t=10^4$.

---
## Section A — Distribution of $N(t)$ *(slides pp. 5–6/54)*

> $P(N(t)=n) = P(N(t)\ge n) - P(N(t)\ge n+1) = P(S_n\le t) - P(S_{n+1}\le t) = F_n(t)-F_{n+1}(t)$,
> where $F_n$ is the $n$-fold convolution of $F$ (the cdf of $S_n$).

### Model
Take **Erlang$(2,\lambda)$ gaps** with $\lambda=2$ (mean $2/\lambda=1$): $S_n$ is a sum of
$2n$ iid $\exp(2)$ variables, so $S_n\sim$ Erlang$(2n,2)$ = Gamma(shape $2n$,
scale $1/2$) and $F_n$ is a Gamma cdf — a rare case where the convolutions are
explicit. For exponential gaps the same formula collapses to the Poisson pmf
(slide p. 6/54).

In [ ]:
lam, k, t = 2.0, 2, 3.0                            # Erlang(k, lam) gaps, mean k/lam = 1; look at N(3)

def F_n(n, t):
    """P(S_n <= t): S_n ~ Erlang(n k, lam) = Gamma(shape n k, scale 1/lam); F_0 = 1."""
    return stats.gamma(n * k, scale=1/lam).cdf(t) if n > 0 else 1.0

ns = np.arange(0, 13)
# TODO 1: the pmf of N(t) for n = 0..12 from P(N(t) = n) = F_n(t) - F_{n+1}(t)   (use F_n above)
pmf = np.full(len(ns), np.nan)
print("P(N(3) = n), n = 0..12:", pmf.round(4), "  sum =", np.nansum(pmf).round(6))

In [ ]:
ref = stats.gamma(6, scale=0.5).cdf(3.0) - stats.gamma(8, scale=0.5).cdf(3.0)   # P(N(3) = 3), computed independently
check("P(N(3) = 3)", None if np.isnan(pmf[3]) else pmf[3], ref, tol=1e-6)
check("sum of the pmf over n = 0..12", None if np.isnan(pmf).any() else pmf.sum(), 1.0, tol=1e-4)

In [ ]:
# --- Simulate: 20,000 paths of the Erlang(2, 2) renewal process; count the events in (0, 3] -----------
n_paths = 20_000
G = rng.gamma(k, 1/lam, (n_paths, 40))                 # 40 gaps per path (more than enough to pass t = 3)
S = G.cumsum(axis=1)
assert (S[:, -1] > t).all()
N3 = (S <= t).sum(axis=1)                              # N(3) on each path
emp = np.bincount(N3, minlength=13)[:13] / n_paths
pmf_ref = np.array([ (stats.gamma(n*k, scale=1/lam).cdf(t) if n > 0 else 1.0) - stats.gamma((n+1)*k, scale=1/lam).cdf(t) for n in ns])   # independent of your TODO

plt.figure()
plt.bar(ns - 0.2, emp, 0.4, label="simulation (20,000 paths)")
plt.bar(ns + 0.2, pmf_ref, 0.4, label=r"$F_n(3) - F_{n+1}(3)$ (Gamma cdfs)")
plt.plot(ns, stats.poisson(lam * t / k).pmf(ns), "ko-", ms=3, lw=1, label="Poisson(3) with the same mean: wrong shape")
plt.xlabel("n"); plt.ylabel("P(N(3) = n)"); plt.legend(fontsize=8)
plt.title("Erlang(2) gaps: the count is more concentrated than a Poisson count"); plt.show()
print("mean N(3): sim", N3.mean().round(3), "  pmf", (ns*pmf_ref).sum().round(3), "  var: sim", N3.var().round(3), "  (Poisson would give var = mean)")

**Think about it** — (i) Erlang(2, 2) renewals are *every second event* of a
$PP(2)$. Use this to write $P(N(t)=n)$ as a sum of two Poisson probabilities and
check it against the Gamma formula. (ii) Why is the variance of $N(3)$ smaller
than its mean here, and when would it be larger? (Hint: the gap cv.)

---
## Section B — The renewal function: three ways to get $m(t)$ *(slides pp. 7–12/54)*

> $m(t)=E[N(t)]=\sum_{n\ge1}F_n(t)$ solves $m(t)=F(t)+\int_0^t m(t-x)\,dF(x)$.
> Example: $F\sim$ uniform$(0,1)$ gives $m'(t)=1+m(t)$, $m(0)=0$, so $m(t)=e^t-1$ for $0\le t\le1$.

### Model
`renewal_function(dist, t)` solves the renewal equation on a grid (no convolution
needed); `renewal_function_mc` averages $N(t)$ over simulated paths; and on
$[0,1]$ the closed form is $e^t-1$ (slide p. 12/54). Beyond $t=1$ the equation
changes piece by piece — the grid solver does not care.

In [ ]:
U = stats.uniform(0, 1)
grid, m_num = renewal_function(U, 3.0)                              # discretised renewal equation on [0, 3]
ts = np.array([0.5, 1.0, 1.5, 2.0, 3.0])
m_mc = renewal_function_mc(sampler_from(U), ts, n_paths=10_000, rng=rng)   # Monte-Carlo E[N(t)]

# TODO 2: the closed form m(t) = e^t - 1 (valid for 0 <= t <= 1) at t = 0.5 and t = 1
m_half = ...
m_one  = ...
print("m(0.5):", m_half, "  m(1):", m_one, "   grid solver:", np.interp([0.5, 1.0], grid, m_num).round(4), "  MC:", m_mc[:2].round(4))

In [ ]:
check("m(0.5) = e^0.5 - 1", m_half, math.exp(0.5) - 1, tol=1e-6)
check("m(1) = e - 1",         m_one,  math.e - 1,         tol=1e-6)

In [ ]:
# --- Visualize: the three routes agree; m(t) - 2t tends to (sigma^2 - mu^2)/(2 mu^2) = -1/3 -----------
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(grid, m_num, lw=2, label="renewal equation on a grid")
gg = grid[grid <= 2.0]
ax[0].plot(gg, m_uniform01(gg), "k--", lw=1.2, label="closed form on [0, 2]")
ax[0].plot(ts, m_mc, "ro", ms=5, label="Monte-Carlo E[N(t)]")
ax[0].plot(grid, 2*grid - 1/3, ":", c="gray", label="asymptote 2t - 1/3")
ax[0].set_xlabel("t"); ax[0].set_ylabel("m(t)"); ax[0].legend(fontsize=8); ax[0].set_title("Uniform(0,1) gaps: m(t) three ways")
ax[1].plot(grid, m_num - 2*grid, lw=2); ax[1].axhline(-1/3, color="k", ls="--", lw=1)
ax[1].set_xlabel("t"); ax[1].set_ylabel("m(t) - t/mu"); ax[1].set_title("m(t) - 2t -> -1/3: the elementary renewal theorem, refined")
plt.tight_layout(); plt.show()

### Final-style · The hyperexponential renewal function *(Companion Proof 2)*

> Compute $m(t)$ when $F(t)=p\,(1-e^{-\mu_1 t})+(1-p)\,(1-e^{-\mu_2 t})$.

Laplace transforms turn the renewal equation into $\tilde m(s)=\tilde F(s)/(1-\tilde F(s))$;
partial fractions give (Companion Proof 2(c))
$$m(t)=\frac{\mu_1\mu_2}{c}\,t+\frac{p(1-p)(\mu_1-\mu_2)^2}{c^2}\big(1-e^{-ct}\big),\qquad c=(1-p)\mu_1+p\mu_2 .$$
Implement it and check it against the grid solver (which knows nothing about
Laplace transforms). Note $\mu_1\mu_2/c=1/E[X]$ — the slope is the long-run rate.

In [ ]:
p, mu1, mu2 = 0.3, 1.0, 5.0
H = hyperexp(p, mu1, mu2)                                  # cdf/pdf/mean/var/rvs of the mixture

def m_closed(t, p, mu1, mu2):
    """Renewal function of the hyperexponential (Laplace-transform solution)."""
    t = np.asarray(t, dtype=float)
    c = ...                                                # TODO 3a: c = (1 - p) mu1 + p mu2
    return np.full_like(t, np.nan)                         # TODO 3b: mu1 mu2 t / c + p (1-p) (mu1 - mu2)^2 / c^2 * (1 - e^{-c t})

grid_h, m_grid = renewal_function(H, 4.0)
print("m(2): closed form", m_closed(2.0, p, mu1, mu2), "  grid solver", np.interp(2.0, grid_h, m_grid).round(4))

In [ ]:
v = float(m_closed(2.0, p, mu1, mu2))
check("m(2), hyperexponential", None if np.isnan(v) else v, float(np.interp(2.0, grid_h, m_grid)), tol=2e-3)
v = float(m_closed(0.3, p, mu1, mu2))
check("m(0.3), hyperexponential", None if np.isnan(v) else v, float(np.interp(0.3, grid_h, m_grid)), tol=2e-3)

In [ ]:
def hyperexp_demo(p=0.3, mu1=1.0, mu2=5.0):
    """m(t) of the hyperexponential: grid solver vs the Laplace closed form (package), and the straight line t/E[X]."""
    Hd = hyperexp(p, mu1, mu2)
    g, mg = renewal_function(Hd, 4.0)
    mu, var = Hd.mean(), Hd.var()
    plt.figure(figsize=(7, 4))
    plt.plot(g, mg, lw=2, label="renewal equation on a grid")
    plt.plot(g, m_hyperexp(g, p, mu1, mu2), "k--", lw=1.2, label="Laplace closed form")
    plt.plot(g, g/mu, ":", c="gray", label="t / E[X]")
    plt.plot(g, g/mu + (var - mu**2)/(2*mu**2), ":", c="red", label="t/E[X] + (sigma^2 - mu^2)/(2 mu^2)")
    plt.xlabel("t"); plt.ylabel("m(t)"); plt.legend(fontsize=8)
    plt.title(f"p={p}, mu1={mu1}, mu2={mu2}: cv = {math.sqrt(var)/mu:.2f}; the transient term is {p*(1-p)*(mu1-mu2)**2/((1-p)*mu1+p*mu2)**2:.3f}"); plt.show()

show(hyperexp_demo, p=FloatSlider(min=.05, max=.95, step=.05, value=.3) if HAS_W else .3,
     mu1=FloatSlider(min=.2, max=3, step=.2, value=1.0) if HAS_W else 1.0,
     mu2=FloatSlider(min=1, max=10, step=.5, value=5.0) if HAS_W else 5.0)

**Think about it** — (i) For $p\to0$ or $p\to1$ the transient term vanishes — why?
(ii) The constant $m(t)-t/\mu\to(\sigma^2-\mu^2)/(2\mu^2)$ is *positive* for the
hyperexponential and $-\tfrac13$ for uniform$(0,1)$. What does its sign say about
the first few renewals compared with a Poisson process of the same rate?

---
## Problem 1 — Battery replacements: the long-run rate *(slide p. 19/54)*

> Kim has a computer mouse that works with a single battery. Whenever the battery
> is out, Kim replaces it with a new one immediately. Given that the lifetime
> distribution of the battery is uniform$(50,100)$, what is the average rate at
> which Kim replaces the battery in the long run?

### Model
Replacements are the renewal epochs of a renewal process with gaps
$X\sim$ uniform$(50,100)$ hours, $\mu=75$. By the limit theorems
(slides pp. 15–18/54) the long-run rate is $1/\mu$ — per sample path and on average.

In [ ]:
a, b = 50.0, 100.0                                      # lifetime uniform(50, 100) hours

# TODO 4: the long-run replacement rate (per hour)
rate = ...
print("long-run replacement rate per hour:", rate)

In [ ]:
check("long-run replacement rate (per hour)", rate, 1/75, tol=1e-9)

In [ ]:
# --- Simulate: N(t)/t for three lifetime distributions with mean 75 -------------------------------
T = 100_000                                                      # hours
three = {"uniform(50,100)": lambda r, n: r.uniform(50, 100, n),
         "exponential, mean 75": lambda r, n: r.exponential(75, n),
         "deterministic 75": lambda r, n: np.full(n, 75.0)}
fig, ax = plt.subplots(figsize=(8, 4))
rate_convergence_plot(three, T=T, rng=rng, ax=ax, mean=75.0)
ax.set_ylim(0.008, 0.02); ax.set_title("N(t)/t -> 1/75 for every shape; the shape sets the wobble"); plt.show()

# the elementary renewal theorem: m(t)/t = E[N(t)]/t over 400 independent paths
ts = np.array([150., 300., 1000., 5000.])
m_hat = renewal_function_mc(three["uniform(50,100)"], ts, n_paths=400, rng=rng)
print("t         :", ts)
print("m(t)/t    :", (m_hat/ts).round(5), "   -> 1/75 =", round(1/75, 5))

In [ ]:
def convergence(cv=0.3, n_rep=5):
    """N(t)/t for lognormal lifetimes with mean 75 and the chosen cv, five paths; band = 1/mu +- 2 sigma / (mu^{3/2} sqrt t)."""
    mu = 75.0; s2 = math.log(1 + cv**2)
    f = lambda r, n: r.lognormal(math.log(mu) - s2/2, math.sqrt(s2), n)
    ts = np.logspace(2, 5, 200)
    plt.figure(figsize=(8, 4))
    for i in range(n_rep):
        S = simulate_renewal(f, 1e5, rng)
        plt.plot(ts, count(S, ts)/ts, lw=1, alpha=.8)
    sigma = cv * mu
    plt.plot(ts, 1/mu + 2*sigma/(mu**1.5*np.sqrt(ts)), "k--", lw=1, label="1/mu +- 2 sigma/(mu^1.5 sqrt t)  (CLT band)")
    plt.plot(ts, 1/mu - 2*sigma/(mu**1.5*np.sqrt(ts)), "k--", lw=1)
    plt.axhline(1/mu, color="k", lw=1)
    plt.xscale("log"); plt.ylim(0.008, 0.02); plt.xlabel("t (hours)"); plt.ylabel("N(t)/t"); plt.legend(fontsize=8)
    plt.title(f"lifetime cv = {cv}: the rate is always 1/75, the time to settle grows like cv^2"); plt.show()

show(convergence, cv=FloatSlider(min=0.05, max=2.0, step=0.05, value=0.3) if HAS_W else 0.3)

**Think about it** — (i) To estimate the rate within $\pm5\%$ with 95 % confidence,
how long must you observe a process with gap cv 1 versus cv 0.2? (Use the band.)
(ii) If a replacement took $D$ hours (random, mean 5) instead of zero, what would
the rate be? Which later section computes the fraction of time *without* a working battery?

---
## Problem 2 — Single-server bank: who gets in? *(slide p. 21/54)*

> Consider a bank with a single server. Customers arrive according to $PP(\lambda)$.
> Each customer enters the bank only when the server is free. Otherwise, the
> customer leaves without entering the bank. The service times of the server
> follow a general distribution with mean $\mu_G$. (a) What is the customers'
> entering rate to the bank? (b) What proportion of arriving customers actually enter the bank?

### Model
A cycle = one service ($Y\sim G$) + the idle period until the next arrival
($X\sim\exp(\lambda)$, memoryless): $E[Z]=\mu_G+1/\lambda$, so
(a) entering rate $=1/E[Z]=\lambda/(1+\lambda\mu_G)$ and
(b) proportion $=$ entering rate$/\lambda = 1/(1+\lambda\mu_G)$ (slide p. 22/54).
Numbers: $\lambda=4$ per hour, $\mu_G=10$ minutes.

In [ ]:
lam, mu_G = 4.0, 10/60                                   # per hour; mean service 10 min = 1/6 h

# TODO 5: (a) entering rate per hour from the cycle E[Z] = mu_G + 1/lam;  (b) proportion = entering rate / lam
rate_in = ...
prop    = ...
print("(a) entering rate:", rate_in, "  (b) proportion:", prop)

In [ ]:
check("(a) entering rate per hour", rate_in, 4/(1 + 4/6), tol=1e-6)
check("(b) proportion entering",   prop,    1/(1 + 4/6), tol=1e-6)

In [ ]:
# --- Simulate the bank: Poisson arrivals, a server that accepts only when free, three service shapes -------
def simulate_bank(arrival_gaps, service_sampler, T, rng):
    """Returns (entering rate, proportion entering). arrival_gaps(rng, n) draws inter-arrival times."""
    A = np.cumsum(arrival_gaps(rng, int(T * 8) + 100)); A = A[A <= T]
    busy_until, entered = 0.0, 0
    for a in A:
        if a >= busy_until:                              # server free: the customer enters
            entered += 1
            busy_until = a + service_sampler(rng, 1)[0]
    return entered / T, entered / len(A)

T = 20_000                                               # hours
pp = lambda r, n: r.exponential(1/lam, n)                # Poisson arrivals, rate lam
services = {"Erlang(2), mean 10 min": lambda r, n: r.gamma(2, mu_G/2, n),
            "deterministic 10 min":   lambda r, n: np.full(n, mu_G),
            "lognormal cv 1.5, mean 10 min": lambda r, n: r.lognormal(math.log(mu_G) - 0.5*math.log(1 + 1.5**2), math.sqrt(math.log(1 + 1.5**2)), n)}
ref_rate, ref_prop = lam/(1 + lam*mu_G), 1/(1 + lam*mu_G)                 # independent of your TODO
print(f"theory: entering rate {ref_rate:.4f}/h, proportion {ref_prop:.4f}   (only the MEAN service time enters)")
for name, g in services.items():
    r_, p_ = simulate_bank(pp, g, T, rng)
    print(f"  Poisson arrivals, service {name:>30}: rate {r_:.4f}  proportion {p_:.4f}")
# now break the Poisson assumption: Erlang(2) arrival gaps with the same rate lam
erl = lambda r, n: r.gamma(2, 1/(2*lam), n)
r_, p_ = simulate_bank(erl, services["Erlang(2), mean 10 min"], T, rng)
print(f"  Erlang(2) arrivals (same rate), Erlang(2) service: rate {r_:.4f}  proportion {p_:.4f}   <- the formula no longer applies")

In [ ]:
def bank_demo(mean_service_min=10.0):
    """Proportion entering as a function of lam (per hour), theory 1/(1 + lam mu_G) vs simulation with Erlang(2) services."""
    mG = mean_service_min / 60
    lams = np.array([0.5, 1, 2, 4, 8, 16])
    sim = [simulate_bank(lambda r, n: r.exponential(1/l, n), lambda r, n: r.gamma(2, mG/2, n), 3000, rng)[1] for l in lams]
    ll = np.linspace(0.2, 16, 200)
    plt.figure(figsize=(7, 4)); plt.plot(ll, 1/(1 + ll*mG), "k", label="1/(1 + lam mu_G)")
    plt.plot(lams, sim, "ro", label="simulation (Erlang(2) services)"); plt.xlabel("arrival rate lam (per hour)"); plt.ylabel("proportion entering")
    plt.legend(fontsize=8); plt.title(f"mean service {mean_service_min:.0f} min: at lam = 4/h, {100/(1+4*mG):.0f}% of arrivals get in"); plt.show()

show(bank_demo, mean_service_min=FloatSlider(min=2, max=30, step=1, value=10) if HAS_W else 10.0)

**Think about it** — (i) Why does the cycle start at a *service completion*
rather than at an arrival, and where exactly is the Poisson (memoryless)
assumption used? (ii) With Erlang(2) arrival gaps the proportion changed although
the arrival rate did not. Which quantity replaces "the idle period is a fresh
$\exp(\lambda)$" — name it with the vocabulary of slide p. 35/54.

---
## Section D — Wald's equation: $N(t)+1$ is a stopping time, $N(t)$ is not *(slides pp. 23–29/54)*

> **Wald:** if $X_1,X_2,\dots$ are iid with finite mean and $N$ is a stopping time
> with $E[N]<\infty$, then $E[\sum_{n=1}^N X_n]=E[N]E[X]$.
> **Corollary:** $E[S_{N(t)+1}]=E[X]\,E[N(t)+1]=\mu\,(m(t)+1)$.

### Model
Uniform$(0,1)$ gaps, $t=3$: estimate $m(3)=E[N(3)]$ from simulated paths and compare
$E[S_{N(t)+1}]$ with Wald's prediction $\mu(m(t)+1)$ — and $E[S_{N(t)}]$ with the
*false* analogue $\mu\,m(t)$. The event $\{N(t)=n\}=\{S_n\le t<S_{n+1}\}$ looks at
$X_{n+1}$, so $N(t)$ is not a stopping time; $N(t)+1$ is (slide p. 29/54).

In [ ]:
t, mu = 3.0, 0.5
n_paths = 40_000
G = rng.uniform(0, 1, (n_paths, 24)); S = G.cumsum(axis=1)
assert (S[:, -1] > t).all()
N = (S <= t).sum(axis=1)                                             # N(t) per path
rows = np.arange(n_paths)
S_next = S[rows, N]                                                  # S_{N(t)+1}: the first epoch after t
S_last = np.where(N > 0, S[rows, np.maximum(N - 1, 0)], 0.0)        # S_{N(t)}: the last epoch before t (0 if none)
m_t = N.mean()                                                       # estimate of m(3)

# TODO 6: Wald's prediction for E[S_{N(t)+1}] in terms of mu and m_t
wald = ...
print("E[S_N(t)+1] sim", S_next.mean().round(4), " Wald:", wald, "    E[S_N(t)] sim", S_last.mean().round(4), " mu m(t):", round(mu*m_t, 4))

In [ ]:
check("E[S_{N(t)+1}] = mu (m(t) + 1)", wald, S_next.mean(), rel=0.01)

In [ ]:
# --- Wald fails without its hypotheses: (1) E[N] = infinity, (2) terms that are not iid -------------------
# (1) symmetric random walk, N = first n with X_1 + ... + X_n = 1: a stopping time, but E[N] = infinity
for n_max in [100, 1_000, 10_000]:
    X = rng.choice([-1, 1], size=(4000, n_max)); W = X.cumsum(axis=1)
    hit = (W == 1).any(axis=1); N_hit = np.where(hit, (W == 1).argmax(axis=1) + 1, n_max)
    print(f"cap {n_max:>6}: P(stopped) = {hit.mean():.3f}   mean of min(N, cap) = {N_hit.mean():8.1f}   E[S_N | stopped] = {W[hit, N_hit[hit]-1].mean():.3f}")
print("  -> the mean of N keeps growing with the cap (E[N] = inf): E[S_N] = 1 != E[N] E[X] = 'inf x 0'.")

# (2) 52 cards, X_i = 1 if the i-th card is an ace, N = position of the 4th ace (homework-style question)
deck = np.array([1]*4 + [0]*48)
Ns = []
for _ in range(20_000):
    d = rng.permutation(deck); Ns.append(np.flatnonzero(d)[3] + 1)
Ns = np.array(Ns)
print(f"cards: E[N] ~ {Ns.mean():.2f} (exact 4*53/5 = 42.4),  sum X_i = 4 always,  E[N] E[X_1] = {Ns.mean()/13:.2f} != 4")
print("  -> N IS a stopping time, but the X_i are not independent (drawing without replacement): Wald does not apply.")

**Think about it** — (i) For the cards, which hypothesis of Wald fails, and would
it hold if each card were drawn *with* replacement? (ii) The random walk hits $+1$
with probability 1 and yet $E[N]=\infty$; explain in one sentence why "$P(N<\infty)=1$"
is not enough for Wald. (iii) Use the corollary to prove
$\liminf_{t\to\infty} m(t)/t\ge 1/\mu$ (Companion Proof 3).

---
## Wrap-up

| question | tool | answer |
|---|---|---|
| $P(N(t)=n)$, Erlang(2) gaps | $F_n(t)-F_{n+1}(t)$ with Gamma cdfs | pmf more concentrated than Poisson |
| $m(t)$, uniform$(0,1)$ | renewal equation $\to$ ODE | $e^t-1$ on $[0,1]$; $m(t)-2t\to-\tfrac13$ |
| $m(t)$, hyperexponential (final-style) | Laplace transform | $\mu_1\mu_2t/c+p(1-p)(\mu_1-\mu_2)^2(1-e^{-ct})/c^2$ |
| battery (#1) | limit theorems | $1/75$ per hour; shape only affects the wobble |
| bank (#2) | cycle = service + idle | $\lambda/(1+\lambda\mu_G)$, $1/(1+\lambda\mu_G)$ (2.4/h, 60 %) |
| Wald | $N(t)+1$ is a stopping time | $E[S_{N(t)+1}]=\mu(m(t)+1)$; not for $N(t)$ |

**Next (CP10):** rewards per cycle — $E[R]/E[X]$ — and what a random observer
sees: age, remaining time, the inspection paradox, and regenerative processes.